# PHYT1D — Module 02: Exercise State Function Φ(u, d, t)

Pure analytical mapping from exercise prescription to time-varying ODE parameter increments.

**Inputs:**
- `u` — fractional exercise intensity [0,1], e.g. 0.50 for 50% VO2max
- `d` — declared bout duration [min]
- `t` — elapsed time since exercise onset [min]
- `t_prime` — elapsed time since bout ended [min] (post-exercise)

**Outputs (ODE increments):**
- ΔSI_ex(t) — insulin sensitivity change during bout
- ΔSI_tail(t′) — post-exercise SI tail
- ΔEGP_res(t) — resistance EGP surge
- ΔFc01_eff(t) — muscle glucose uptake increment
- kempt_eff(t) — effective gastric emptying rate

**References:** Riddell 2017 [2]; Hinshaw 2013 [12]; Yardley 2013 [11]; Dalla Man 2009 [18]


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from phyt1d_fixed_params import FIXED   # imported from 01_ode_base context


In [ ]:
# ─── Re-import fixed params if running standalone ────────────────────────────
FIXED = {
    "VI": 0.126, "ke": 0.127, "beta": 8.0,
    "f": 0.90, "VG": 1.45, "alpha": 7.0,
    "r1": 1.44, "r2": 0.81, "Gth": 60.0,
    "Fc01": 0.0097, "gamma_aer": 0.003, "gamma_res": 0.0015,
    "tau_ramp": 20.0, "delta_res": 0.003,
}


## 2.1 ΔSI_ex — Insulin Sensitivity Change During Exercise Bout

In [ ]:
def delta_SI_ex(u, t, exercise_type, beta_aer, tau_on, delta_res=None, tau_ramp=None,
               p=FIXED):
    """
    Aerobic:   ΔSI_ex(t) = beta_aer · u · (1 − exp(−t / tau_on))
    Resistance: ΔSI_ex(t) = −delta_res · u · min(t / tau_ramp, 1)

    Parameters
    ----------
    u             : float — fractional intensity [0,1]
    t             : float — elapsed time since exercise onset [min]
    exercise_type : str   — 'aerobic' | 'resistance'
    beta_aer      : float — aerobic SI gain coefficient (Group B)
    tau_on        : float — SI ramp-up time constant [min] (Group B)
    delta_res     : float — resistance SI reduction coefficient (Group D, optional override)
    tau_ramp      : float — resistance ramp-up time [min] (Group D, optional override)

    Returns
    -------
    float — ΔSI_ex(t)
    """
    if delta_res is None:
        delta_res = p["delta_res"]
    if tau_ramp is None:
        tau_ramp = p["tau_ramp"]

    if exercise_type == "aerobic":
        return beta_aer * u * (1.0 - np.exp(-t / tau_on))
    elif exercise_type == "resistance":
        return -delta_res * u * min(t / tau_ramp, 1.0)
    else:
        raise ValueError(f"Unknown exercise_type: '{exercise_type}'. Use 'aerobic' or 'resistance'.")


## 2.2 ΔSI_tail — Post-Exercise Insulin Sensitivity Tail

In [ ]:
def delta_SI_tail(u, t_prime, beta_aer, tau_post):
    """
    Post-exercise SI tail (both aerobic and resistance types).

    ΔSI_tail(t′) = beta_aer · u · exp(−t′ / tau_post)

    Primary driver of late-onset hypoglycaemia (4–12 h duration).

    Parameters
    ----------
    u        : float — fractional exercise intensity [0,1]
    t_prime  : float — time since exercise bout ended [min]
    beta_aer : float — aerobic SI gain coefficient (Group B)
    tau_post : float — post-exercise SI decay time constant [min] (Group B)

    Returns
    -------
    float — ΔSI_tail(t′)
    """
    return beta_aer * u * np.exp(-t_prime / tau_post)


## 2.3 ΔEGP_res — Resistance Exercise EGP Surge

In [ ]:
def delta_EGP_res(u, t, exercise_type, beta_res, tau_ramp=None, p=FIXED):
    """
    Catecholamine-driven hepatic glucose surge during resistance exercise.

    ΔEGP_res(t) = beta_res · u² · min(t / tau_ramp, 1)   [resistance]
    ΔEGP_res(t) = 0                                        [aerobic]

    Quadratic u² captures disproportionate EGP at high intensity:
    u=0.3 → 0.09·beta_res  vs  u=0.8 → 0.64·beta_res  (7-fold difference).

    Parameters
    ----------
    u             : float — fractional intensity [0,1]
    t             : float — elapsed time since onset [min]
    exercise_type : str   — 'aerobic' | 'resistance'
    beta_res      : float — resistance EGP gain coefficient (Group B)
    tau_ramp      : float — ramp-up time [min] (Group D)

    Returns
    -------
    float — ΔEGP_res(t) [mmol/kg/min]
    """
    if tau_ramp is None:
        tau_ramp = p["tau_ramp"]
    if exercise_type == "resistance":
        return beta_res * (u ** 2) * min(t / tau_ramp, 1.0)
    return 0.0


## 2.4 ΔFc01_eff — Non-Insulin-Mediated Muscle Uptake

In [ ]:
def Fc01_eff(u, t, exercise_type, tau_ramp=None, p=FIXED):
    """
    Effective non-insulin-mediated glucose uptake (brain + erythrocytes + exercising muscle).

    Fc01_eff(t) = Fc01 + gamma_aer · u · min(t / tau_ramp, 1)   [aerobic]
    Fc01_eff(t) = Fc01 + gamma_res · u · min(t / tau_ramp, 1)   [resistance]

    gamma_aer > gamma_res reflecting higher uptake per intensity during aerobic exercise.
    Baseline Fc01 = 0.0097 mmol/kg/min (Hovorka 2004).

    Parameters
    ----------
    u             : float — fractional intensity [0,1]
    t             : float — elapsed time since onset [min]
    exercise_type : str   — 'aerobic' | 'resistance'
    tau_ramp      : float — ramp-up time [min] (Group D)
    p             : dict  — fixed parameters

    Returns
    -------
    float — Fc01_eff(t) [mmol/kg/min]
    """
    if tau_ramp is None:
        tau_ramp = p["tau_ramp"]
    ramp = min(t / tau_ramp, 1.0)
    if exercise_type == "aerobic":
        return p["Fc01"] + p["gamma_aer"] * u * ramp
    else:
        return p["Fc01"] + p["gamma_res"] * u * ramp


## 2.5 kempt_eff — Gastric Emptying Acceleration

In [ ]:
def kempt_eff(kempt, u, phi):
    """
    Exercise accelerates gastric emptying during light-to-moderate bouts.

    kempt_eff(t) = kempt · (1 + phi · u)

    phi ∈ [0, 0.5] identified per patient (Group B).
    At u=0.5, kempt increases up to 25%, shifting Ra(t) peak 15–20 min earlier.

    Parameters
    ----------
    kempt : float — baseline gastric emptying rate [min⁻¹] (Group A)
    u     : float — fractional intensity [0,1]
    phi   : float — gastric emptying acceleration factor (Group B)

    Returns
    -------
    float — effective kempt [min⁻¹]
    """
    return kempt * (1.0 + phi * u)


## 2.6 SI_eff — Composite Effective Insulin Sensitivity

In [ ]:
def SI_eff(SI, u, t, t_prime, d, exercise_type, theta_B, p=FIXED):
    """
    SI_eff(t) = SI · [1 + ΔSI_ex(t) + ΔSI_tail(t′)]

    Combines baseline SI with in-bout and post-bout modulations.

    Parameters
    ----------
    SI            : float — basal insulin sensitivity (Group A)
    u             : float — fractional intensity [0,1]
    t             : float — current simulation time [min]
    t_prime       : float — time since bout ended [min]; 0 during bout
    d             : float — bout duration [min]
    exercise_type : str   — 'aerobic' | 'resistance'
    theta_B       : dict  — Group B params: beta_aer, tau_on, tau_post
    p             : dict  — fixed params

    Returns
    -------
    float — SI_eff [mL/μU·min]
    """
    beta_aer = theta_B["beta_aer"]
    tau_on   = theta_B["tau_on"]
    tau_post = theta_B["tau_post"]

    dSI_ex   = delta_SI_ex(u, t, exercise_type, beta_aer, tau_on, p=p)
    dSI_tail = delta_SI_tail(u, t_prime, beta_aer, tau_post)

    return SI * (1.0 + dSI_ex + dSI_tail)


## 2.7 Master Φ(u, d, t) Function

In [ ]:
def Phi(u, d, t_sim, t_start, exercise_type, theta_A, theta_B, p=FIXED):
    """
    Master exercise state function Φ(u, d, t).
    Returns all ODE parameter increments for time t_sim.

    Parameters
    ----------
    u             : float — fractional VO2max intensity [0,1]
    d             : float — bout duration [min]
    t_sim         : float — simulation time [min] from sim start
    t_start       : float — exercise start time [min] from sim start
    exercise_type : str   — 'aerobic' | 'resistance'
    theta_A       : dict  — Group A params: SI, kempt, ...
    theta_B       : dict  — Group B params: beta_aer, tau_on, tau_post, phi, ...
    p             : dict  — fixed params

    Returns
    -------
    dict with keys:
        SI_eff      — effective insulin sensitivity [mL/μU·min]
        delta_EGP   — EGP increment [mmol/kg/min]
        Fc01_eff    — effective muscle uptake [mmol/kg/min]
        kempt_eff   — effective gastric emptying rate [min⁻¹]
        active      — bool: is exercise bout currently active?
        post_active — bool: is post-exercise tail active?
    """
    t_end = t_start + d

    # Time within bout / post-bout
    if t_start <= t_sim <= t_end:
        t_ex      = t_sim - t_start   # elapsed since onset
        t_prime   = 0.0
        active    = True
        post_active = False
    elif t_sim > t_end:
        t_ex      = d
        t_prime   = t_sim - t_end     # elapsed since bout ended
        active    = False
        post_active = True
    else:
        # Before exercise starts
        t_ex = 0.0; t_prime = 0.0
        active = False; post_active = False

    SI      = theta_A["SI"]
    kempt_b = theta_A["kempt"]
    phi     = theta_B.get("phi", 0.0)

    if u == 0.0:
        # Rest — no exercise effects
        return {
            "SI_eff"     : SI,
            "delta_EGP"  : 0.0,
            "Fc01_eff"   : p["Fc01"],
            "kempt_eff"  : kempt_b,
            "active"     : False,
            "post_active": False,
        }

    SI_e = SI_eff(SI, u, t_ex, t_prime, d, exercise_type, theta_B, p)
    dEGP = delta_EGP_res(u, t_ex, exercise_type, theta_B["beta_res"], p=p)
    Fc01 = Fc01_eff(u, t_ex, exercise_type, p=p)
    ke   = kempt_eff(kempt_b, u if active else 0.0, phi)

    return {
        "SI_eff"     : SI_e,
        "delta_EGP"  : dEGP,
        "Fc01_eff"   : Fc01,
        "kempt_eff"  : ke,
        "active"     : active,
        "post_active": post_active,
    }


## 2.8 Visualisation — SI_eff Profile During and After Exercise

In [ ]:
# Reproduce Figure 7: Post-exercise SI_eff for three tau_post values
t_arr = np.linspace(-20, 420, 1000)
d_bout = 60.0
t_start_vis = 0.0

theta_A_ex = {"SI": 1.0, "kempt": 0.055}   # normalised SI=1 for relative plot

tau_post_cases = [
    (60,  "τ_post = 60 min (low fitness)",      "tomato"),
    (150, "τ_post = 150 min (moderate fitness)", "goldenrod"),
    (270, "τ_post = 270 min (high fitness)",     "teal"),
]

plt.figure(figsize=(10, 5))
for tau_p, label, col in tau_post_cases:
    theta_B_ex = {"beta_aer": 0.4, "tau_on": 15.0, "tau_post": tau_p,
                  "beta_res": 0.005, "phi": 0.0}
    si_vals = []
    for t in t_arr:
        phi_out = Phi(u=0.5, d=d_bout, t_sim=t, t_start=t_start_vis,
                      exercise_type="aerobic",
                      theta_A=theta_A_ex, theta_B=theta_B_ex)
        si_vals.append(phi_out["SI_eff"])
    plt.plot(t_arr, si_vals, color=col, lw=2, label=label)

plt.axvline(0,  color="steelblue", ls="--", lw=1.2, label="Exercise onset")
plt.axvline(d_bout, color="steelblue", ls=":",  lw=1.2, label="Bout ends")
plt.axhline(1.0, color="grey", ls="--", lw=1, label="Baseline SI")
plt.xlabel("Time from exercise onset [min]", fontsize=12)
plt.ylabel("SI_eff / SI_base (relative)", fontsize=12)
plt.title("Figure 7 — Post-Exercise SI_eff Profile (50% VO₂max, 60-min aerobic bout)", fontsize=13)
plt.legend(fontsize=10)
plt.tight_layout()
plt.savefig("phi_SI_profile.png", dpi=150)
plt.show()
print("✓ Φ module validated. Figure saved.")
